In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
image_dir = '/content/drive/MyDrive/XAI_Project/results_multi_cam/Models/'
print(f"Updated image directory to: {image_dir}")

Updated image directory to: /content/drive/MyDrive/XAI_Project/results_multi_cam/Models/


In [ ]:
from pathlib import Path
import numpy as np
import re

# Define base directory (if not already defined in previous cells)
base_results_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/Models/')

def load_data_for_metrics_corrected(base_dir):
    all_loaded_data = {}
    # Iterate through top-level folders (e.g., efficientnet_b0, LIME Explanations)
    for top_level_folder in base_dir.iterdir():
        if not top_level_folder.is_dir():
            continue

        model_name = top_level_folder.name
        all_loaded_data[model_name] = {}

        # Check if this top_level_folder itself contains .npy files.
        # This covers cases where a "method" like LIME might be a top-level folder
        # containing saliency maps directly, without a sub-method folder.
        npy_files_directly_in_folder = list(top_level_folder.glob('*.npy'))

        if npy_files_directly_in_folder:
            # If .npy files are found directly, treat the folder's name as the method name
            # within this 'model'. This handles "LIME Explanations" as a "model" where
            # the explanations are directly within that folder.
            method_name_for_direct_files = model_name # Use folder name as method name
            all_loaded_data[model_name][method_name_for_direct_files] = {}
            for saliency_map_path in npy_files_directly_in_folder:
                try:
                    match_image_id = re.match(r'(cat_\d+|dog_\d+)', saliency_map_path.name)
                    image_id = match_image_id.group(0) if match_image_id else saliency_map_path.stem.split('_')[0]
                    saliency_map = np.load(saliency_map_path)
                    all_loaded_data[model_name][method_name_for_direct_files][image_id] = saliency_map
                except Exception as e:
                    print(f"⚠️ Error loading {saliency_map_path}: {e}")

        # Now, iterate through sub-folders, which are typically method folders
        for method_folder in top_level_folder.iterdir():
            if method_folder.is_dir(): # Ensure it's a directory (subfolder)
                method_name = method_folder.name
                # Initialize the method_name dictionary if it's not already from direct files
                if method_name not in all_loaded_data[model_name]:
                    all_loaded_data[model_name][method_name] = {}

                # Iterate through ALL .npy files in the method folder
                for saliency_map_path in method_folder.glob('*.npy'): # Changed from *_raw.npy
                    try:
                        match_image_id = re.match(r'(cat_\d+|dog_\d+)', saliency_map_path.name)
                        image_id = match_image_id.group(0) if match_image_id else saliency_map_path.stem.split('_')[0]
                        saliency_map = np.load(saliency_map_path)
                        all_loaded_data[model_name][method_name][image_id] = saliency_map
                    except Exception as e:
                        print(f"⚠️ Error loading {saliency_map_path}: {e}")
    return all_loaded_data

# Call the function to load all data
print(f"Loading saliency map data from: {base_results_dir}")
all_loaded_data = load_data_for_metrics_corrected(base_results_dir)

# Confirmation message
print("\nSaliency map data loaded successfully!")
print(f"Loaded models: {list(all_loaded_data.keys())}")

# Optional: Display a sample of loaded data keys for verification
if all_loaded_data:
    sample_model = list(all_loaded_data.keys())[0]
    if all_loaded_data[sample_model]:
        sample_method = list(all_loaded_data[sample_model].keys())[0]
        print(f"Sample data structure for model '{sample_model}', method '{sample_method}':")
        print(f"  First 5 image IDs: {list(all_loaded_data[sample_model][sample_method].keys())[:5]}")
    else:
        print(f"No methods found for model '{sample_model}'.")
else:
    print("No data was loaded. Please check the `base_results_dir` path and folder structure.")

print("\nVerifying specific methods:")
requested_methods = ['vanilla_saliency', 'smoothgrad', 'integrated_gradients', 'LIME'] # Keep LIME as requested by user
found_requested_methods = {meth: False for meth in requested_methods}

for model_name, methods_data in all_loaded_data.items():
    # Check if the model_name itself matches one of the requested methods (e.g., LIME Explanations)
    for req_meth in requested_methods:
        if req_meth.lower() in model_name.lower() and methods_data: # Check if it's a model that *has* loaded data
            print(f"- Found model '{model_name}' (matching '{req_meth}') with {len(methods_data.keys())} method(s) and first method '{list(methods_data.keys())[0]}' having {len(list(methods_data.values())[0])} images.")
            found_requested_methods[req_meth] = True

    for method_name in methods_data.keys():
        for req_meth in requested_methods:
            if req_meth.lower() in method_name.lower() and methods_data[method_name]:
                print(f"- Found method '{method_name}' under model '{model_name}' (matching '{req_meth}') with {len(methods_data[method_name])} image(s).")
                found_requested_methods[req_meth] = True

for req_meth, found in found_requested_methods.items():
    if not found:
        print(f"- Method '{req_meth}' was NOT found.")

Loading saliency map data from: /content/drive/MyDrive/XAI_Project/results_multi_cam/Models

Saliency map data loaded successfully!
Loaded models: ['efficientnet_b0', 'efficientnet_b3', 'mobilenet_v3_large', 'resnet18', 'squeezenet']
Sample data structure for model 'efficientnet_b0', method 'GradCAM':
  First 5 image IDs: ['cat_1374', 'cat_1375', 'cat_1378', 'cat_1379', 'cat_1382']

Verifying specific methods:
- Found method 'LIME' under model 'efficientnet_b0' (matching 'LIME') with 594 image(s).
- Found method 'vanilla_saliency' under model 'efficientnet_b0' (matching 'vanilla_saliency') with 598 image(s).
- Found method 'integrated_gradients' under model 'efficientnet_b0' (matching 'integrated_gradients') with 598 image(s).
- Found method 'smoothgrad' under model 'efficientnet_b0' (matching 'smoothgrad') with 598 image(s).
- Found method 'LIME' under model 'efficientnet_b3' (matching 'LIME') with 598 image(s).
- Found method 'vanilla_saliency' under model 'efficientnet_b3' (matching

In [ ]:
if 'all_loaded_data' in locals() and all_loaded_data:
    print("Załadowane modele:")
    for model_name in all_loaded_data.keys():
        print(f"- {model_name}")
elif 'all_loaded_data' in locals() and not all_loaded_data:
    print("Zmienna 'all_loaded_data' jest pusta. Nie załadowano żadnych modeli.")
else:
    print("Zmienna 'all_loaded_data' nie została jeszcze zdefiniowana. Upewnij się, że komórka ładująca dane została uruchomiona.")

Załadowane modele:
- efficientnet_b0
- efficientnet_b3
- mobilenet_v3_large
- resnet18
- squeezenet


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit
from scipy.stats import pearsonr # For comparison/fallback, though Numba implementation will be used

# 2. Define calculate_pcc function with Numba optimization
@jit(nopython=True)
def calculate_pcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba.")

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit, cuda # Import cuda for GPU optimization
from scipy.stats import pearsonr

# CUDA kernel for element-wise calculations (numerator parts, sum of squared deviations parts)
@cuda.jit
def _pcc_parts_kernel(arr1_d, arr2_d, mean1, mean2, numerator_parts_d, dev1_parts_d, dev2_parts_d):
    idx = cuda.grid(1)
    if idx < arr1_d.size:
        val1 = arr1_d[idx] - mean1
        val2 = arr2_d[idx] - mean2
        numerator_parts_d[idx] = val1 * val2
        dev1_parts_d[idx] = val1 * val1
        dev2_parts_d[idx] = val2 * val2

# Define GPU-optimized calculate_pcc function
def calculate_pcc_gpu(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Ensure data is float32 for CUDA compatibility
    flat_arr1 = arr1.flatten().astype(np.float32)
    flat_arr2 = arr2.flatten().astype(np.float32)

    if len(flat_arr1) != len(flat_arr2):
        return 0.0

    # Calculate means on CPU. For very large arrays, this could also be parallelized on GPU.
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Transfer data to device (GPU)
    arr1_d = cuda.to_device(flat_arr1)
    arr2_d = cuda.to_device(flat_arr2)

    # Allocate device arrays for intermediate results
    numerator_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)
    dev1_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)
    dev2_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)

    # Configure and launch kernel for element-wise operations
    threadsperblock = 256
    blockspergrid = (flat_arr1.size + (threadsperblock - 1)) // threadsperblock
    _pcc_parts_kernel[blockspergrid, threadsperblock](arr1_d, arr2_d, mean1, mean2,
                                                      numerator_parts_d, dev1_parts_d, dev2_parts_d)
    cuda.synchronize() # Wait for the kernel to complete

    # Copy intermediate results back to host for sum reduction.
    # For ultimate GPU performance, these sums could also be performed on the GPU using
    # `numba.cuda.reduce` or custom reduction kernels.
    numerator = numerator_parts_d.copy_to_host().sum()
    sum_sq_dev1 = dev1_parts_d.copy_to_host().sum()
    sum_sq_dev2 = dev2_parts_d.copy_to_host().sum()

    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0
    else:
        pcc = numerator / denominator
        if np.isnan(pcc):
            return 0.0
        return pcc

# Keep the original CPU-optimized version as a fallback or for explicit comparison
@jit(nopython=True)
def calculate_pcc_cpu(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

# Decide which version to use based on CUDA availability
try:
    if cuda.is_available():
        calculate_pcc = calculate_pcc_gpu
        print("Using GPU-optimized `calculate_pcc`.")
    else:
        calculate_pcc = calculate_pcc_cpu
        print("CUDA not available. Falling back to CPU-optimized `calculate_pcc`.")
except Exception as e:
    print(f"Error checking CUDA availability: {e}. Falling back to CPU-optimized `calculate_pcc`.")
    calculate_pcc = calculate_pcc_cpu

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba (GPU or CPU fallback).")

start_time = time.time()

pcc_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating PCC
            if map_a.shape == map_b.shape:
                pcc_value = calculate_pcc(map_a, map_b)
                pcc_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'PCC': pcc_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping PCC for this pair.")

end_time = time.time()
total_calculation_time = end_time - start_time

print(f"Total PCC calculation time: {total_calculation_time:.2f} seconds")

# Convert results to DataFrame
df_pcc_all_models = pd.DataFrame(pcc_all_models_results)

# Save to CSV
output_csv_filename = '/content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_full_matrix_results.csv'
df_pcc_all_models.to_csv(output_csv_filename, index=False)

print(f"PCC full matrix results for all models and method comparisons saved to {output_csv_filename}")
print("First 5 rows of the results DataFrame:")
display(df_pcc_all_models.head())

In [ ]:
output_csv_filename = '/content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_full_matrix_results.csv'
df_pcc_all_models.to_csv(output_csv_filename, index=False)

print(f"PCC full matrix results for all models and method comparisons saved to {output_csv_filename}")
print("First 5 rows of the results DataFrame:")
display(df_pcc_all_models.head())

PCC full matrix results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_full_matrix_results.csv
First 5 rows of the results DataFrame:


,model,image_id,method_a,method_b,PCC
0,efficientnet_b0,cat_1001,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3443,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_2412,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_1612,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2883,GradCAM,GradCAM,1.0


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from skimage.metrics import structural_similarity as ssim # Import SSIM

# Function to calculate SSIM, expecting 2D arrays
def calculate_ssim(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Ensure both arrays have the same shape
    if arr1.shape != arr2.shape:
        return 0.0 # Or raise an error

    # Normalize arrays to a 0-1 range. This is crucial for SSIM to have a consistent data_range.
    # Saliency maps often have values that aren't strictly 0-1 initially.
    # We normalize each map independently based on its own min/max.
    # Add a small epsilon to avoid division by zero if max-min is zero (flat map)
    epsilon = 1e-8

    if arr1.max() - arr1.min() > epsilon:
        normalized_arr1 = (arr1 - arr1.min()) / (arr1.max() - arr1.min())
    else:
        normalized_arr1 = np.zeros_like(arr1) # If map is flat, normalize to all zeros

    if arr2.max() - arr2.min() > epsilon:
        normalized_arr2 = (arr2 - arr2.min()) / (arr2.max() - arr2.min())
    else:
        normalized_arr2 = np.zeros_like(arr2) # If map is flat, normalize to all zeros

    # Calculate SSIM. data_range=1.0 because we normalized to 0-1.
    # If the maps are 3D (e.g., RGB), channel_axis might be needed.
    # Assuming saliency maps are grayscale (2D) or will be treated as such.
    ssim_value = ssim(normalized_arr1, normalized_arr2, data_range=1.0)
    return ssim_value

print("SSIM function defined.")

start_time_ssim = time.time()

ssim_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating SSIM
            if map_a.shape == map_b.shape:
                ssim_value = calculate_ssim(map_a, map_b)
                ssim_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'SSIM': ssim_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping SSIM for this pair.")

end_time_ssim = time.time()
total_calculation_time_ssim = end_time_ssim - start_time_ssim

print(f"Total SSIM calculation time: {total_calculation_time_ssim:.2f} seconds")

# Convert results to DataFrame
df_ssim_all_models = pd.DataFrame(ssim_all_models_results)

# Save to CSV in the same directory as PCC results
output_csv_filename_ssim = '/content/drive/MyDrive/XAI_Project/results_multi_cam/ssim_all_models_full_matrix_results.csv'
df_ssim_all_models.to_csv(output_csv_filename_ssim, index=False)

print(f"SSIM full matrix results for all models and method comparisons saved to {output_csv_filename_ssim}")
print("First 5 rows of the SSIM results DataFrame:")
display(df_ssim_all_models.head())

SSIM function defined.
Total SSIM calculation time: 320.83 seconds
SSIM full matrix results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/ssim_all_models_full_matrix_results.csv
First 5 rows of the SSIM results DataFrame:


,model,image_id,method_a,method_b,SSIM
0,efficientnet_b0,cat_1001,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3443,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_2412,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_1612,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2883,GradCAM,GradCAM,1.0


In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from scipy.stats import spearmanr # Import Spearman's correlation

# Function to calculate Spearman's Rank Correlation Coefficient
def calculate_spearmanr(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # spearmanr returns (correlation_coefficient, p_value)
    corr, _ = spearmanr(flat_arr1, flat_arr2)

    # Handle potential NaN values (e.g., if one array has zero variance or all values are the same)
    if np.isnan(corr):
        return 0.0
    return corr

print("Spearman's Rank Correlation function defined.")

Spearman's Rank Correlation function defined.


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from scipy.stats import spearmanr # Import Spearman's correlation

# Function to calculate Spearman's Rank Correlation Coefficient
def calculate_spearmanr(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # spearmanr returns (correlation_coefficient, p_value)
    corr, _ = spearmanr(flat_arr1, flat_arr2)

    # Handle potential NaN values (e.g., if one array has zero variance or all values are the same)
    if np.isnan(corr):
        return 0.0
    return corr

print("Spearman's Rank Correlation function defined.")

start_time_spearmanr = time.time()

spearmanr_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating Spearman's
            if map_a.shape == map_b.shape:
                spearmanr_value = calculate_spearmanr(map_a, map_b)
                spearmanr_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'SpearmanR': spearmanr_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping SpearmanR for this pair.")

end_time_spearmanr = time.time()
total_calculation_time_spearmanr = end_time_spearmanr - start_time_spearmanr

print(f"Total Spearman's Rank Correlation calculation time: {total_calculation_time_spearmanr:.2f} seconds")

# Convert results to DataFrame
df_spearmanr_all_models = pd.DataFrame(spearmanr_all_models_results)

# Save to CSV in the same directory as PCC results
output_csv_filename_spearmanr = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/spearmanr_all_models_full_matrix_results.csv'
df_spearmanr_all_models.to_csv(output_csv_filename_spearmanr, index=False)

print(f"Spearman's Rank Correlation full matrix results for all models and method comparisons saved to {output_csv_filename_spearmanr}")
print("First 5 rows of the Spearman's R results DataFrame:")
display(df_spearmanr_all_models.head())

Spearman's Rank Correlation function defined.


/tmp/ipython-input-2197119555.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(flat_arr1, flat_arr2)


Total Spearman's Rank Correlation calculation time: 1736.51 seconds
Spearman's Rank Correlation full matrix results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/spearmanr_all_models_full_matrix_results.csv
First 5 rows of the Spearman's R results DataFrame:


,model,image_id,method_a,method_b,SpearmanR
0,efficientnet_b0,cat_1025,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3435,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_1043,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_276,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2126,GradCAM,GradCAM,1.0


In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from sklearn.metrics import matthews_corrcoef # Import MCC
from scipy.special import rel_entr # For KL Divergence

# Function to calculate Matthews Correlation Coefficient
def calculate_mcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(flat_arr1 == flat_arr1[0]): # Check if arr1 is constant
        bin_arr1 = np.zeros_like(flat_arr1, dtype=bool)
    else:
        median_arr1 = np.median(flat_arr1)
        bin_arr1 = flat_arr1 > median_arr1

    if np.all(flat_arr2 == flat_arr2[0]): # Check if arr2 is constant
        bin_arr2 = np.zeros_like(flat_arr2, dtype=bool)
    else:
        median_arr2 = np.median(flat_arr2)
        bin_arr2 = flat_arr2 > median_arr2

    # matthews_corrcoef raises ValueError if input contains only one class
    # or if the confusion matrix leads to division by zero (e.g., all True or all False)
    if len(np.unique(bin_arr1)) < 2 or len(np.unique(bin_arr2)) < 2:
        return 0.0 # MCC is undefined in such cases, return 0.0 as a neutral value

    try:
        mcc = matthews_corrcoef(bin_arr1, bin_arr2)
        return mcc
    except ValueError: # Catch cases where MCC cannot be computed (e.g., all predictions are the same)
        return 0.0 # Return 0.0 for undefined MCC

# Function to calculate KL Divergence
def calculate_kl_divergence(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    p = arr1.flatten()
    q = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(p) != len(q):
        return np.nan # Or raise an error

    # Normalize to form probability distributions
    # Add a small epsilon to avoid division by zero when normalizing or taking log
    epsilon = 1e-10
    p = p + epsilon
    q = q + epsilon

    p_sum = np.sum(p)
    q_sum = np.sum(q)

    if p_sum == 0 or q_sum == 0:
        return np.nan # Cannot calculate KL if one distribution sums to zero

    p_normalized = p / p_sum
    q_normalized = q / q_sum

    # Calculate KL Divergence using scipy.special.rel_entr
    # rel_entr(p, q) calculates p * log(p/q) element-wise
    kl_div = np.sum(rel_entr(p_normalized, q_normalized))
    return kl_div

print("Matthews Correlation Coefficient (MCC) and Kullback-Leibler (KL) Divergence functions defined.")

Matthews Correlation Coefficient (MCC) and Kullback-Leibler (KL) Divergence functions defined.


In [ ]:
start_time_mcc = time.time()

mcc_all_models_results = []
kl_div_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating metrics
            if map_a.shape == map_b.shape:
                # Calculate MCC
                mcc_value = calculate_mcc(map_a, map_b)
                mcc_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'MCC': mcc_value
                })

                # Calculate KL Divergence
                kl_div_value = calculate_kl_divergence(map_a, map_b)
                kl_div_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'KL_Divergence': kl_div_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping MCC and KL for this pair.")

end_time_mcc = time.time()
total_calculation_time_mcc = end_time_mcc - start_time_mcc

print(f"Total MCC and KL Divergence calculation time: {total_calculation_time_mcc:.2f} seconds")

# Convert MCC results to DataFrame and save to CSV
df_mcc_all_models = pd.DataFrame(mcc_all_models_results)
output_csv_filename_mcc = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/mcc_all_models_full_matrix_results.csv'
df_mcc_all_models.to_csv(output_csv_filename_mcc, index=False)
print(f"MCC results for all models and method comparisons saved to {output_csv_filename_mcc}")
print("First 5 rows of the MCC results DataFrame:")
display(df_mcc_all_models.head())

# Convert KL Divergence results to DataFrame and save to CSV
df_kl_div_all_models = pd.DataFrame(kl_div_all_models_results)
output_csv_filename_kl = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/kl_divergence_all_models_full_matrix_results.csv'
df_kl_div_all_models.to_csv(output_csv_filename_kl, index=False)
print(f"KL Divergence results for all models and method comparisons saved to {output_csv_filename_kl}")
print("First 5 rows of the KL Divergence results DataFrame:")
display(df_kl_div_all_models.head())

Total MCC and KL Divergence calculation time: 3821.64 seconds
MCC results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/mcc_all_models_full_matrix_results.csv
First 5 rows of the MCC results DataFrame:


,model,image_id,method_a,method_b,MCC
0,efficientnet_b0,cat_1025,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3435,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_1043,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_276,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2126,GradCAM,GradCAM,1.0


KL Divergence results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/kl_divergence_all_models_full_matrix_results.csv
First 5 rows of the KL Divergence results DataFrame:


,model,image_id,method_a,method_b,KL_Divergence
0,efficientnet_b0,cat_1025,GradCAM,GradCAM,0.0
1,efficientnet_b0,cat_3435,GradCAM,GradCAM,0.0
2,efficientnet_b0,cat_1043,GradCAM,GradCAM,0.0
3,efficientnet_b0,cat_276,GradCAM,GradCAM,0.0
4,efficientnet_b0,cat_2126,GradCAM,GradCAM,0.0
